# 07 — Final Evaluation

**Goal:** One-time evaluation on the locked test set. These are your real numbers.

> ⚠️ **This is the first and only time we use `test.csv`.**  
> We get ONE shot. Do not re-run this notebook to tune further — that's test set leakage.

**Steps:**
1. Load train + val + test (all splits)
2. Retrain best model on train + val combined (more data = better)
3. Predict on test set
4. Compute all final metrics
5. Diagnostic visualizations
6. Log model with `mlflow.sklearn.log_model()`
7. Automatic evaluation with `mlflow.evaluate()`

In [ ]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import json
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.base import BaseEstimator, TransformerMixin

try:
    from lightgbm import LGBMRegressor
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

from sklearn.ensemble import GradientBoostingRegressor

sns.set_theme(style='whitegrid')
PROCESSED_PATH    = '../data/processed/'
MLFLOW_EXPERIMENT = 'energy-demand-prediction'
TARGET            = 'total load actual'

## 1. Load ALL Data — Train + Val + Test

In [ ]:
train_raw = pd.read_csv(PROCESSED_PATH + 'train.csv')
val_raw   = pd.read_csv(PROCESSED_PATH + 'val.csv')
test_raw  = pd.read_csv(PROCESSED_PATH + 'test.csv')  # 🔓 UNLOCKED

for df in [train_raw, val_raw, test_raw]:
    df['time'] = pd.to_datetime(df['time'], utc=True)

with open(PROCESSED_PATH + 'feature_config.json') as f:
    config = json.load(f)

with open(PROCESSED_PATH + 'best_params.json') as f:
    best_params = json.load(f)

with open(PROCESSED_PATH + 'best_model.txt') as f:
    BEST_MODEL_NAME = f.read().strip()

ALL_FEATURES = config['all_features']

print(f'Train: {len(train_raw):,} rows  |  {train_raw["time"].min().date()} → {train_raw["time"].max().date()}')
print(f'Val:   {len(val_raw):,} rows  |  {val_raw["time"].min().date()} → {val_raw["time"].max().date()}')
print(f'Test:  {len(test_raw):,} rows  |  {test_raw["time"].min().date()} → {test_raw["time"].max().date()}')
print()
print(f'Model:  {BEST_MODEL_NAME}')
print(f'Params: {best_params}')

## 2. Feature Transformer

In [ ]:
class EnergyFeatureTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, lag_hours=None, add_cyclical=True):
        self.lag_hours=lag_hours or [1,24,48,168,336]; self.add_cyclical=add_cyclical
    def fit(self, X, y=None): return self
    def transform(self, X):
        df=X.copy(); df['time']=pd.to_datetime(df['time'],utc=True)
        df=df.sort_values('time').reset_index(drop=True)
        df=self._cal(df); df=self._cyc(df) if self.add_cyclical else df
        df=self._lag(df); df=self._roll(df); df=self._weather(df)
        return df.dropna().reset_index(drop=True)
    def _cal(self, df):
        t=df['time']; df['hour']=t.dt.hour; df['dow']=t.dt.dayofweek
        df['month']=t.dt.month; df['day_of_year']=t.dt.dayofyear
        df['week_of_year']=t.dt.isocalendar().week.astype(int)
        df['quarter']=t.dt.quarter; df['is_weekend']=(t.dt.dayofweek>=5).astype(int)
        df['season']=df['month'].map({12:3,1:3,2:3,3:0,4:0,5:0,6:1,7:1,8:1,9:2,10:2,11:2})
        try:
            import holidays; es=holidays.Spain(years=range(2015,2020))
            df['is_holiday']=t.dt.date.astype(str).map(lambda d:1 if d in [str(h) for h in es] else 0)
        except: df['is_holiday']=0
        return df
    def _cyc(self, df):
        df['hour_sin']=np.sin(2*np.pi*df['hour']/24); df['hour_cos']=np.cos(2*np.pi*df['hour']/24)
        df['dow_sin']=np.sin(2*np.pi*df['dow']/7);   df['dow_cos']=np.cos(2*np.pi*df['dow']/7)
        df['month_sin']=np.sin(2*np.pi*df['month']/12); df['month_cos']=np.cos(2*np.pi*df['month']/12)
        df['doy_sin']=np.sin(2*np.pi*df['day_of_year']/365); df['doy_cos']=np.cos(2*np.pi*df['day_of_year']/365)
        return df
    def _lag(self, df):
        for lag in self.lag_hours: df[f'lag_{lag}h']=df[TARGET].shift(lag)
        return df
    def _roll(self, df):
        s=df[TARGET].shift(1)
        df['rolling_mean_24h']=s.rolling(24,min_periods=12).mean()
        df['rolling_mean_168h']=s.rolling(168,min_periods=84).mean()
        df['rolling_std_24h']=s.rolling(24,min_periods=12).std()
        df['rolling_max_24h']=s.rolling(24,min_periods=12).max()
        df['rolling_min_24h']=s.rolling(24,min_periods=12).min()
        return df
    def _weather(self, df):
        df['temp_squared']=df['temp']**2; df['is_cold']=(df['temp']<280).astype(int)
        df['is_hot']=(df['temp']>298).astype(int); df['temp_humidity']=df['temp']*df['humidity']/100
        df['is_raining']=(df['rain_1h']>0).astype(int); df['wind_chill']=df['wind_speed']*(1-df['temp']/310)
        return df

fe = EnergyFeatureTransformer()
print('Transformer ready ✅')

## 3. Retrain on Train + Val Combined

More training data → better final model for deployment.  
We concatenate train + val, apply features, then train.

In [ ]:
# Concatenate train + val for final training
# Sort is critical — lag features require chronological order
trainval_raw = pd.concat([train_raw, val_raw]).sort_values('time').reset_index(drop=True)

trainval_fe = fe.fit_transform(trainval_raw)
X_trainval  = trainval_fe[ALL_FEATURES]
y_trainval  = trainval_fe[TARGET]

print(f'Train+Val shape: {X_trainval.shape}')
print(f'Date range: {trainval_fe["time"].min().date()} → {trainval_fe["time"].max().date()}')

# Build best model
if BEST_MODEL_NAME == 'LightGBM' and HAS_LGB:
    best_model = LGBMRegressor(**best_params)
else:
    # Fallback to GradientBoosting if LightGBM not installed
    gb_params = {k: v for k, v in best_params.items()
                 if k in ['n_estimators','max_depth','learning_rate','subsample',
                          'min_samples_split','min_samples_leaf','random_state']}
    best_model = GradientBoostingRegressor(**gb_params)

best_pipeline = Pipeline([('model', best_model)])
best_pipeline.fit(X_trainval, y_trainval)

print(f'\n✅ Model retrained on {len(X_trainval):,} rows')

## 4. Predict on Test Set

In [ ]:
# For test: lag features need context from the end of trainval
# We prepend the last 336 rows of trainval to provide lag context
CONTEXT_ROWS = 336  # max lag used
context      = trainval_raw.tail(CONTEXT_ROWS)
test_with_context = pd.concat([context, test_raw]).sort_values('time').reset_index(drop=True)

test_fe  = fe.transform(test_with_context)

# Filter to only actual test period (remove context rows)
test_start = test_raw['time'].min()
test_fe    = test_fe[test_fe['time'] >= test_start].reset_index(drop=True)

X_test = test_fe[ALL_FEATURES]
y_test = test_fe[TARGET]

print(f'X_test: {X_test.shape}')
print(f'y_test: {y_test.shape}')
print(f'Test period: {test_fe["time"].min().date()} → {test_fe["time"].max().date()}')

y_pred = best_pipeline.predict(X_test)
print(f'\nPredictions generated: {len(y_pred):,} rows')

## 5. Final Metrics

In [ ]:
rmse  = np.sqrt(mean_squared_error(y_test, y_pred))
mae   = mean_absolute_error(y_test, y_pred)
r2    = r2_score(y_test, y_pred)
mape  = np.mean(np.abs((y_test.values - y_pred) / y_test.values)) * 100
rmse_pct = rmse / y_test.mean() * 100

# Peak hour RMSE (hours 8-11 and 19-21 — high stakes periods)
peak_mask = test_fe['hour'].isin([8,9,10,11,19,20,21])
peak_rmse = np.sqrt(mean_squared_error(y_test[peak_mask], y_pred[peak_mask]))

# Weekend vs weekday
weekend_mask = test_fe['is_weekend'] == 1
wd_rmse  = np.sqrt(mean_squared_error(y_test[~weekend_mask], y_pred[~weekend_mask]))
we_rmse  = np.sqrt(mean_squared_error(y_test[weekend_mask],  y_pred[weekend_mask]))

print('FINAL TEST SET RESULTS')
print('=' * 50)
print(f'  RMSE:             {rmse:>10,.2f} MW')
print(f'  RMSE (% of mean): {rmse_pct:>10.2f} %')
print(f'  MAE:              {mae:>10,.2f} MW')
print(f'  R²:               {r2:>10.4f}')
print(f'  MAPE:             {mape:>10.2f} %')
print()
print(f'  Peak hour RMSE:   {peak_rmse:>10,.2f} MW  (hours 8-11, 19-21)')
print(f'  Weekday RMSE:     {wd_rmse:>10,.2f} MW')
print(f'  Weekend RMSE:     {we_rmse:>10,.2f} MW')
print()
print(f'  Target mean:      {y_test.mean():>10,.2f} MW')
print(f'  Target std:       {y_test.std():>10,.2f} MW')

## 6. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
residuals = y_test.values - y_pred

# ── Plot 1: Full test period — actual vs predicted ──
axes[0, 0].plot(test_fe['time'], y_test.values, color='steelblue',
                 linewidth=0.6, alpha=0.9, label='Actual')
axes[0, 0].plot(test_fe['time'], y_pred, color='tomato',
                 linewidth=0.6, alpha=0.7, label='Predicted')
axes[0, 0].set_title(f'Actual vs Predicted — Full Test Period  |  RMSE: {rmse:,.0f} MW')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('MW')
axes[0, 0].legend()
axes[0, 0].xaxis.set_major_locator(mdates.MonthLocator())
axes[0, 0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(axes[0, 0].xaxis.get_majorticklabels(), rotation=20)

# ── Plot 2: Scatter actual vs predicted ──
axes[0, 1].scatter(y_test.values, y_pred, alpha=0.3, s=5, color='steelblue')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0, 1].plot(lims, lims, 'r--', lw=2, label='Perfect')
axes[0, 1].set_title(f'Scatter: Actual vs Predicted  |  R² = {r2:.4f}')
axes[0, 1].set_xlabel('Actual MW')
axes[0, 1].set_ylabel('Predicted MW')
axes[0, 1].legend()

# ── Plot 3: Residuals over time ──
axes[1, 0].plot(test_fe['time'], residuals, color='tomato', linewidth=0.5, alpha=0.7)
axes[1, 0].axhline(y=0, color='black', linestyle='--', lw=1)
axes[1, 0].axhline(y=rmse, color='orange', linestyle=':', lw=1, label=f'+RMSE ({rmse:,.0f})')
axes[1, 0].axhline(y=-rmse, color='orange', linestyle=':', lw=1, label=f'-RMSE')
axes[1, 0].set_title('Residuals Over Time  |  Random scatter = good model')
axes[1, 0].set_xlabel('Date')
axes[1, 0].set_ylabel('Residual (MW)')
axes[1, 0].legend()
axes[1, 0].xaxis.set_major_locator(mdates.MonthLocator())
axes[1, 0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(axes[1, 0].xaxis.get_majorticklabels(), rotation=20)

# ── Plot 4: Residuals by hour of day ──
test_fe_copy      = test_fe.copy()
test_fe_copy['residual'] = residuals
hourly_residuals  = test_fe_copy.groupby('hour')['residual'].agg(['mean','std'])
axes[1, 1].bar(hourly_residuals.index, hourly_residuals['mean'], color='steelblue', edgecolor='white')
axes[1, 1].errorbar(hourly_residuals.index, hourly_residuals['mean'],
                      yerr=hourly_residuals['std'], fmt='none', color='black', capsize=3)
axes[1, 1].axhline(y=0, color='red', linestyle='--', lw=1)
axes[1, 1].set_title('Mean Residual by Hour  |  Should be ~0 at every hour')
axes[1, 1].set_xlabel('Hour of Day')
axes[1, 1].set_ylabel('Mean Residual (MW)')

plt.suptitle(f'Final Model Evaluation — {BEST_MODEL_NAME} (Tuned)  |  RMSE={rmse:,.0f} MW  R²={r2:.4f}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PROCESSED_PATH + 'final_eval_plots.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Plots saved')

In [ ]:
# Feature importance
model_step = best_pipeline.named_steps['model']
if hasattr(model_step, 'feature_importances_'):
    importances = pd.Series(
        model_step.feature_importances_,
        index=ALL_FEATURES
    ).sort_values(ascending=True).tail(20)

    fig, ax = plt.subplots(figsize=(10, 8))
    colors = ['seagreen' if 'lag' in f or 'rolling' in f else
              'steelblue' if any(w in f for w in ['temp','humid','wind','rain','cloud']) else
              'gold' for f in importances.index]
    importances.plot(kind='barh', ax=ax, color=colors)
    ax.set_title(f'Top 20 Feature Importances — {BEST_MODEL_NAME}  |  🟩 lag/roll  🟦 weather  🟨 calendar')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.savefig(PROCESSED_PATH + 'feature_importances.png', bbox_inches='tight', dpi=150)
    plt.show()

In [ ]:
# Zoom in on 2 weeks of predictions
zoom_start = test_fe['time'].iloc[0]
zoom_end   = zoom_start + pd.Timedelta(days=14)
zoom_mask  = (test_fe['time'] >= zoom_start) & (test_fe['time'] <= zoom_end)

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(test_fe['time'][zoom_mask], y_test.values[zoom_mask],
        color='steelblue', linewidth=1.5, label='Actual', marker='o', markersize=2)
ax.plot(test_fe['time'][zoom_mask], y_pred[zoom_mask],
        color='tomato', linewidth=1.5, label='Predicted', linestyle='--')
ax.set_title('2-Week Zoom: Actual vs Predicted')
ax.set_xlabel('Date')
ax.set_ylabel('MW')
ax.legend()
ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%a %d'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 7. Log to MLflow + mlflow.evaluate()

In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT)

with mlflow.start_run(run_name='final_model_evaluation'):

    # ── Log params ──
    mlflow.log_param('model_type',      BEST_MODEL_NAME)
    mlflow.log_param('tuned',           True)
    mlflow.log_param('train_on',        'train + val combined')
    mlflow.log_param('n_features',      len(ALL_FEATURES))
    mlflow.log_params({f'hp_{k}': v for k, v in best_params.items()})

    # ── Log final metrics ──
    mlflow.log_metric('test_rmse',      rmse)
    mlflow.log_metric('test_rmse_pct',  rmse_pct)
    mlflow.log_metric('test_mae',       mae)
    mlflow.log_metric('test_r2',        r2)
    mlflow.log_metric('test_mape',      mape)
    mlflow.log_metric('peak_rmse',      peak_rmse)
    mlflow.log_metric('weekday_rmse',   wd_rmse)
    mlflow.log_metric('weekend_rmse',   we_rmse)

    # ── Log plots ──
    mlflow.log_artifact(PROCESSED_PATH + 'final_eval_plots.png')
    mlflow.log_artifact(PROCESSED_PATH + 'feature_importances.png')

    # ── Log the trained pipeline ──
    model_info = mlflow.sklearn.log_model(
        sk_model=best_pipeline,
        artifact_path='final_pipeline',
        registered_model_name='energy_demand_model',
        input_example=X_test.head(5)
    )

    print(f'✅ Model logged to MLflow')
    print(f'   URI: {model_info.model_uri}')
    print()

    # ── mlflow.evaluate() — automatic metrics + artifacts ──
    # Creates: residuals plot, actual-vs-predicted, all standard regression metrics
    eval_data = X_test.copy()
    eval_data['target'] = y_test.values

    eval_results = mlflow.evaluate(
        model=model_info.model_uri,
        data=eval_data,
        targets='target',
        model_type='regressor',
        evaluators='default'
    )

    print('mlflow.evaluate() auto-logged metrics:')
    for k, v in eval_results.metrics.items():
        print(f'  {k}: {v:.4f}')

## 8. Final Project Summary

In [ ]:
print('=' * 60)
print('PROJECT COMPLETE — ENERGY DEMAND PREDICTION')
print('=' * 60)
print(f'  Model:             {BEST_MODEL_NAME} (tuned with Optuna)')
print(f'  Features:          {len(ALL_FEATURES)} features')
print(f'  Training data:     2015-01 → 2018-05 (train + val)')
print(f'  Test period:       2018-05 → 2018-12')
print()
print('FINAL TEST METRICS:')
print(f'  RMSE:   {rmse:>10,.2f} MW   ({rmse_pct:.1f}% of mean demand)')
print(f'  MAE:    {mae:>10,.2f} MW')
print(f'  R²:     {r2:>10.4f}')
print(f'  MAPE:   {mape:>10.2f} %')
print()
print('BASELINE COMPARISON:')
print(f'  Naive baseline (lag_168h): ~3,767 MW RMSE  (check MLflow for exact)')
print(f'  Ridge baseline:            ~3,078 MW RMSE  (check MLflow for exact)')
print(f'  Final model:               {rmse:,.0f} MW RMSE  ← this run')
print()
print('NEXT STEPS:')
print('  → Extract notebook logic into src/ scripts')
print('  → Write tests/ with pytest')
print('  → Build FastAPI prediction endpoint')
print('  → Dockerize')
print('  → CI/CD with GitHub Actions')
print()
print('All runs visible at: mlflow ui → http://127.0.0.1:5000')